In [1]:
import numpy as np
import pandas as pd

\begin{align*}
h_\theta(x)=g(\theta^{T}x)=\frac{1}{1+e^{-\theta^{T}x}}\\
g(z)=\frac{1}{1+e^{-z}}\\
g(z)'=g(z)(1-g(z))\\
p(y|x;\theta)=(h_\theta(x))^{y}(1-h_\theta(x))^{1-y}\\
L(\theta)=\prod_{i=1}^{m}p(y^{(i)}|x^{(i)};\theta)=\\
=\prod_{i=1}^{m}(h_\theta(x^{(i)})^{y^{(i)}})(1-h_\theta(x^{(i)}))^{1-y^{(i)}}\\
l(\theta)=\log L(\theta)=\sum_{i=1}^{m}y^{(i)}\log h(x^{(i)})+(1-y^{(i)})\log (1-h_\theta(x^{(i)}))\\
J(\theta)=-\frac{1}{m}\sum_{i=1}^{m}y^{(i)}\log h(x^{(i)})+(1-y^{(i)})\log (1-h_\theta(x^{(i)}))
\end{align*}

\begin{align*}
\frac{\partial}{\partial \theta_j}l(\theta)=\\
g(\theta^{T}x)(1-g(\theta^{T}x))y\frac{1}{g(\theta^{T}x)}x_j-g(\theta^{T}x)(1-g(\theta^{T}x))(1-y)\frac{1}{1-g(\theta^{T}x)}x_j=\\
(y-h_\theta(x))x_j \\
\nabla_\theta J(\theta)=-\frac{1}{m}x^{T}(y-h_\theta(x))
\end{align*}
So update for $\theta$ is: $\theta := \theta+\alpha x^{T}(y-h_\theta(x))$\\
where shape of $x$ is $(m,n)$, $y$ is $(m,1)$, $h_\theta(x))$ is $(m,1)$ and we end up with $(n,1)$ which is shape of $\theta$.



In [20]:
class LogisticRegression:
    def __init__(self,alpha=0.01,reg=0,eps=1e-5,max_iter=200):
        self.alpha=alpha
        self.reg=reg
        self.eps=eps
        self.max_iter=max_iter
    def sigmoid(self,z):
        z = np.clip(z, -500, 500)
        return 1/(1+np.exp(-z))
    def fit(self,X,y):
        m,n=X.shape
        self.theta=np.zeros(n)
        it=0
        prev_theta=self.theta+5*self.eps
        
        while it<self.max_iter and np.linalg.norm(prev_theta-self.theta,ord=1)>self.eps:
            it+=1
            prev_theta=self.theta.copy()
            penalty=self.reg*self.theta
            penalty[0]=0.0
            self.theta=self.theta+self.alpha/m*X.T@(y-self.sigmoid(X@self.theta))-penalty
        if it<self.max_iter:
            print(f"Converged in {it} iterations")
        else:
            print(f"Failed to converge in {it} iterations")
    def predict(self,x):
        return self.sigmoid(x@self.theta)>=0.5

In [3]:
df=pd.read_csv('data/iris.csv')

In [4]:
X,y=df.loc[:99,['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']],df.loc[:99,'Species']

In [5]:
y.unique()

<StringArray>
['Iris-setosa', 'Iris-versicolor']
Length: 2, dtype: str

In [6]:
y=y.map({'Iris-setosa':0,'Iris-versicolor':1})

In [7]:
X.insert(0,'x0',np.ones(X.shape[0]))

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [9]:
X_train_std=X_train.iloc[:,1:].std(axis=0)
X_train_mean=X_train.iloc[:,1:].mean(axis=0)
X_train.iloc[:,1:]=(X_train.iloc[:,1:]-X_train_mean)/X_train_std
X_test.iloc[:,1:]=(X_test.iloc[:,1:]-X_train_mean)/X_train_std

In [22]:
lr=LogisticRegression(max_iter=1000,alpha=0.1)
lr.fit(X_train.values,y_train.values)
pred=lr.predict(X_test)
print(f"accuracy: {np.mean(pred==y_test)}")

Failed to converge in 1000 iterations


np.float64(1.0)

So the dataset is linearly separable so adding l2 regularization should help with convergence

In [21]:
lr=LogisticRegression(max_iter=1000,alpha=0.1,reg=0.1)
lr.fit(X_train.values,y_train.values)
pred=lr.predict(X_test)
print(f"accuracy: {np.mean(pred==y_test)}")

Converged in 121 iterations


np.float64(1.0)